# Confirmatory Analysis 04 — Temporal Stability and Autocorrelation

**Goal:** Verify whether the log(EUR) distribution is stable over time (no drift), and whether log-returns carry autocorrelation — which determines what temporal features belong in the model.

**Tables:** gold_price_features (full history)

**α = 0.05, Bonferroni correction within each hypothesis family.**

---
## Hypotheses
1. The log(EUR) distribution is stable over time — no drift between the first and second half of the history
2. Market log-returns have significant autocorrelation at lag 1 (prices have short-term memory)
3. Log-return autocorrelation differs between Tier 1 (<€100) and Tier 3 (>€1000)

⚠️ **Correction from cda_01:** Ljung-Box tested only up to lag max n//5 (not 30!). At n=90 → max lag = 18.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import ks_2samp

In [2]:
gold = duckdb.connect("../../data/gold/cards.duckdb", read_only=True)

In [3]:
df = gold.execute("""
    SELECT uuid, snapshot_date, eur
    FROM gold_price_features
    WHERE eur IS NOT NULL AND uuid IS NOT NULL
    ORDER BY uuid, snapshot_date
""").df()
df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
df["log_eur"] = np.log1p(df["eur"])
# Per-card daily log-return: log1p(EUR[t]) - log1p(EUR[t-1])
df["log_return_1d"] = df.groupby("uuid")["log_eur"].diff(1)

# Market-wide daily median — one representative series
market = (
    df.groupby("snapshot_date")["eur"]
    .median()
    .reset_index()
    .rename(columns={"eur": "median_eur"})
    .set_index("snapshot_date")
    .sort_index()
)
market["log_eur"] = np.log1p(market["median_eur"])
market["log_return"] = market["log_eur"].diff(1)

n_snapshots = df["snapshot_date"].nunique()
date_min_ts = df["snapshot_date"].min()
date_max_ts = df["snapshot_date"].max()

print(f"Cards:         {df['uuid'].nunique():,}")
print(f"Snapshots:     {n_snapshots}  ({date_min_ts.date()} – {date_max_ts.date()})")
print(f"Total rows:    {len(df):,}")
print("\nMarket-wide series:")
print(market[["median_eur", "log_eur", "log_return"]].round(6).to_string())

if n_snapshots < 30:
    print(
        f"\n⚠ INSUFFICIENT DATA: {n_snapshots} snapshots (30 required for ADF/Ljung-Box)."
    )
    print(
        "  H1, H2, H3 code paths handle this gracefully with expected-result documentation."
    )

Cards:         82,413
Snapshots:     5  (2026-06-04 – 2026-06-08)
Total rows:    412,065

Market-wide series:
               median_eur   log_eur  log_return
snapshot_date                                  
2026-06-04           0.27  0.239017         NaN
2026-06-05           0.27  0.239017         0.0
2026-06-06           0.27  0.239017         0.0
2026-06-07           0.27  0.239017         0.0
2026-06-08           0.27  0.239017         0.0

⚠ INSUFFICIENT DATA: 5 snapshots (30 required for ADF/Ljung-Box).
  H1, H2, H3 code paths handle this gracefully with expected-result documentation.


## H1 — Distribution Stability Over Time (KS Test)

**Hypothesis:** The distribution of median log(EUR) per card is the same in the first and second half of the price history (no price level drift).

**Test:** Two-sample Kolmogorov-Smirnov. Samples: median log_eur per card in the first half vs. the second half of history.

**Why:** If the distribution drifts → temporal train/test split must be strict (older data for training, newer for testing). A stable distribution is still split temporally (to prevent leakage), but drift is not an additional concern.

**Interpretation:** H₀ rejected = drift exists. Watch D (max distance between CDFs) — at large n even D=0.02 gives p<0.05. D < 0.05 = practically stable, D > 0.10 = meaningful drift.

In [4]:
mid_date = date_min_ts + (date_max_ts - date_min_ts) / 2

first_half = (
    df[df["snapshot_date"] <= mid_date].groupby("uuid")["log_eur"].median().dropna()
)
second_half = (
    df[df["snapshot_date"] > mid_date].groupby("uuid")["log_eur"].median().dropna()
)

n_days_first = df[df["snapshot_date"] <= mid_date]["snapshot_date"].nunique()
n_days_second = df[df["snapshot_date"] > mid_date]["snapshot_date"].nunique()

print("=== H1: Distribution Stability (Two-Sample KS) ===")
print(f"History: {date_min_ts.date()} → {date_max_ts.date()}")
print(f"Split:   {mid_date.date()}")
print(f"First half:  {n_days_first} day(s), {len(first_half):,} cards")
print(f"Second half: {n_days_second} day(s), {len(second_half):,} cards")
print()

MIN_DAYS_HALF = 15
if n_days_first < MIN_DAYS_HALF or n_days_second < MIN_DAYS_HALF:
    print(
        f"INSUFFICIENT DATA: each half needs ≥{MIN_DAYS_HALF} days to produce meaningful median estimates."
    )
    print(f"  First half: {n_days_first} days, second half: {n_days_second} days.")
    rerun_date = (date_max_ts + pd.Timedelta(days=30)).date()
    print(f"  Re-run after: approximately {rerun_date} (≥30 daily snapshots).")
    print()
    print("Methodology (auto-executes when data is available):")
    print("  1. Compute per-card median log_eur in each half")
    print("  2. Two-sample KS test: H0 = same distribution")
    print("  3. D < 0.05 = practically stable; D > 0.10 = meaningful drift")
    print(
        "  4. Implication: drift → tighten temporal split; stable → standard split OK"
    )
    ks_stat, p_h1 = np.nan, np.nan

else:
    ks_stat, p_h1 = ks_2samp(first_half.values, second_half.values)
    stability = (
        "PRACTICALLY STABLE (D < 0.05)"
        if ks_stat < 0.05
        else (
            "AMBIGUOUS (0.05 ≤ D < 0.10)"
            if ks_stat < 0.10
            else "DRIFT DETECTED (D ≥ 0.10)"
        )
    )
    print(f"KS D = {ks_stat:.4f},  p = {p_h1:.4e}")
    print(f"Stability: {stability}")
    print(
        f"Decision (α=0.05): {'Distribution drift — tighten split' if p_h1 < 0.05 else 'H0 not rejected — no significant drift'}"
    )

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(
        first_half,
        bins=60,
        alpha=0.6,
        color="steelblue",
        density=True,
        label=f"First half ({n_days_first} days)",
    )
    ax.hist(
        second_half,
        bins=60,
        alpha=0.6,
        color="tomato",
        density=True,
        label=f"Second half ({n_days_second} days)",
    )
    ax.set_xlabel("log1p(EUR) — per-card median in period")
    ax.set_ylabel("Density")
    ax.set_title(f"H1 — Distribution stability  (KS D={ks_stat:.4f}, p={p_h1:.3e})")
    ax.legend()
    plt.tight_layout()
    plt.show()

=== H1: Distribution Stability (Two-Sample KS) ===
History: 2026-06-04 → 2026-06-08
Split:   2026-06-06
First half:  3 day(s), 82,413 cards
Second half: 2 day(s), 82,413 cards

INSUFFICIENT DATA: each half needs ≥15 days to produce meaningful median estimates.
  First half: 3 days, second half: 2 days.
  Re-run after: approximately 2026-07-08 (≥30 daily snapshots).

Methodology (auto-executes when data is available):
  1. Compute per-card median log_eur in each half
  2. Two-sample KS test: H0 = same distribution
  3. D < 0.05 = practically stable; D > 0.10 = meaningful drift
  4. Implication: drift → tighten temporal split; stable → standard split OK


**H1 observations:**
- Only 3 snapshots (1–2 days per half) — KS test cannot be run meaningfully (need ≥15 days per half to produce stable per-card median estimates).
- The test auto-executes once sufficient data is available: the same code will run the KS test, plot the overlaid histograms, and report D and p.
- **Temporal split is still required regardless** of H1 outcome: even with a stable distribution, training on future data would introduce target leakage.
- **Re-run after:** approximately 2026-07-04 (≥30 daily snapshots → ≥15 per half).

## H2 — Log-Return Autocorrelation (Ljung-Box)

**Hypothesis:** Market-wide log-returns have significant autocorrelation at lag 1.

**Test:** Ljung-Box Q-test. Tested lags: [1, 7, 14] or fewer if n//5 < 14.

**CRITICAL RULE:** max_lag = n // 5. At n=90 → max lag = 18. Do not test lags above n//5 — results are unreliable for small samples.

**Implication:** If lag-1 is significant → past returns predict future returns → feature `price_change_1d_pct` (= lag-1 log-return) belongs in the model. If not significant → prices are locally efficient → no sub-7-day historical signal.

**Effect size:** ACF(1) = correlation between day t and day t-1. |ACF(1)| > 0.3 = strong, practically meaningful autocorrelation.

In [5]:
returns = market["log_return"].dropna()
n_ret = len(returns)
max_lag = n_ret // 5

print("=== H2: Log-Return Autocorrelation (Ljung-Box) ===")
print(f"Market log-returns: n={n_ret}")
print(f"Max lag (n//5 rule): {max_lag}  (need ≥10 for reliable results → n≥50)")
print()

MIN_N_LB = 50
if n_ret < MIN_N_LB:
    print(f"INSUFFICIENT DATA: n={n_ret} < {MIN_N_LB} required.")
    print(f"  max_lag = {max_lag} at n={n_ret} → no testable lags (need max_lag ≥ 1).")
    rerun_lb = (date_max_ts + pd.Timedelta(days=50)).date()
    print(f"  Re-run after: approximately {rerun_lb} (≥50 daily snapshots).")
    print()
    if n_ret >= 3:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            acf_1_val = returns.autocorr(lag=1)
        print(f"  ACF(1) = {acf_1_val:.4f}  ⚠  not reliable at n={n_ret}")
    else:
        print(f"  ACF(1) = N/A  (n={n_ret} < 3)")
    print()
    print("Expected result at ≥50 snapshots:")
    print(
        "  Ljung-Box lag-1: Q=?, p=?  (significant if p<0.05 → price_change_1d_pct is predictive)"
    )
    print("  ACF(1) > 0.3 = strong momentum → include lag-1 feature in model")
    print("  ACF(7) > 0.3 = weekly momentum → include lag-7 feature")

else:
    test_lags = [lag for lag in [1, 7, 14] if lag <= max_lag]
    lb = acorr_ljungbox(returns, lags=test_lags, return_df=True)
    lb.index = test_lags
    lb.index.name = "lag"
    print(lb.round(4))

    acf_1 = returns.autocorr(lag=1)
    acf_7 = returns.autocorr(lag=7) if n_ret > 7 else float("nan")
    p_lag1 = lb.loc[1, "lb_pvalue"] if 1 in lb.index else float("nan")

    print(
        f"\nACF(1) = {acf_1:.4f} — {'STRONG' if abs(acf_1) > 0.3 else 'weak'} autocorrelation"
    )
    if not np.isnan(acf_7):
        print(
            f"ACF(7) = {acf_7:.4f} — {'STRONG' if abs(acf_7) > 0.3 else 'weak'} autocorrelation"
        )
    print(
        f"Feature price_change_1d_pct: {'PREDICTIVE ✓' if p_lag1 < 0.05 else 'not confirmed'}"
    )

    max_lag_acf = min(max_lag, 30)
    fig, ax = plt.subplots(figsize=(10, 4))
    plot_acf(returns, lags=max_lag_acf, alpha=0.05, ax=ax)
    ax.set_title(f"ACF — market log-returns  (n={n_ret}, max_lag={max_lag_acf})")
    plt.tight_layout()
    plt.show()

=== H2: Log-Return Autocorrelation (Ljung-Box) ===
Market log-returns: n=4
Max lag (n//5 rule): 0  (need ≥10 for reliable results → n≥50)

INSUFFICIENT DATA: n=4 < 50 required.
  max_lag = 0 at n=4 → no testable lags (need max_lag ≥ 1).
  Re-run after: approximately 2026-07-28 (≥50 daily snapshots).

  ACF(1) = nan  ⚠  not reliable at n=4

Expected result at ≥50 snapshots:
  Ljung-Box lag-1: Q=?, p=?  (significant if p<0.05 → price_change_1d_pct is predictive)
  ACF(1) > 0.3 = strong momentum → include lag-1 feature in model
  ACF(7) > 0.3 = weekly momentum → include lag-7 feature


**H2 observations:**
- n=2 log-return observations (3 snapshots → 2 differences) — Ljung-Box requires n≥50; max_lag = 2//5 = 0 → no lags testable.
- ACF(1) is undefined/unreliable at n=2 (zero variance in a 2-point series).
- The code auto-executes the full Ljung-Box test and ACF plot once ≥50 snapshots are available.
- **Re-run after:** approximately 2026-07-26 (≥50 daily snapshots).

## H3 — Log-Return Autocorrelation Differs Between Tier 1 and Tier 3

**Hypothesis:** Tier 3 (>€1000) cards have stronger log-return autocorrelation than Tier 1 (<€100).

**Rationale:** Expensive cards (Power Nine, Reserved List) are subject to speculative momentum — price increases can trigger further increases (positive autocorrelation). Bulk common cards are more efficiently priced.

**Method:** Compute ACF(1) for per-tier daily median log-returns. Compare values — at small n we do not apply formal difference tests (insufficient power).

**If Tier 3 has stronger autocorrelation:** The lag-1 feature matters more for expensive cards → a Tier 3 model can benefit from autoregression where a Tier 1 model cannot.

In [6]:
# Assign tier by latest snapshot price for each card
latest_price = (
    df.sort_values("snapshot_date").groupby("uuid")[["eur"]].last().reset_index()
)
latest_price["tier"] = pd.cut(
    latest_price["eur"],
    bins=[0, 100, 1000, np.inf],
    labels=[1, 2, 3],
)
df_tier = df.merge(latest_price[["uuid", "tier"]], on="uuid", how="left")

print("=== H3: Log-Return ACF(1) by Price Tier ===")
print()

MIN_N_ACF = 5
tier_results = []
for tier_val in [1, 2, 3]:
    tier_returns = (
        df_tier[df_tier["tier"] == tier_val]
        .groupby("snapshot_date")["log_return_1d"]
        .median()
        .sort_index()
        .dropna()
    )
    n_tier = len(tier_returns)

    if n_tier < MIN_N_ACF:
        print(
            f"Tier {tier_val}: n={n_tier} return observations — insufficient (need ≥{MIN_N_ACF})"
        )
        tier_results.append(
            {
                "tier": tier_val,
                "n_returns": n_tier,
                "acf_1": np.nan,
                "verdict": "INSUFFICIENT DATA",
            }
        )
        continue

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        acf_1_tier = tier_returns.autocorr(lag=1)

    strength = "STRONG (|r|>0.3)" if abs(acf_1_tier) > 0.3 else "WEAK (|r|≤0.3)"
    print(f"Tier {tier_val}: n={n_tier} days, ACF(1) = {acf_1_tier:.4f}  ({strength})")
    tier_results.append(
        {
            "tier": tier_val,
            "n_returns": n_tier,
            "acf_1": round(acf_1_tier, 4),
            "verdict": strength,
        }
    )

print()
all_insufficient = all(r["n_returns"] < MIN_N_ACF for r in tier_results)
if all_insufficient:
    rerun_h3 = (date_max_ts + pd.Timedelta(days=15)).date()
    print(f"All tiers insufficient at n_snapshots={n_snapshots}.")
    print(
        f"Re-run after: approximately {rerun_h3} (≥15 daily snapshots for initial ACF)."
    )
    print()
    print("Expected result:")
    print(
        "  Tier 3 (>€1000): higher |ACF(1)| — speculative momentum in expensive cards"
    )
    print("  Tier 1 (<€100):  lower |ACF(1)| — more liquid, more efficient bulk market")
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
    for ax, tier_val in zip(axes, [1, 2, 3]):
        tier_ret = (
            df_tier[df_tier["tier"] == tier_val]
            .groupby("snapshot_date")["log_return_1d"]
            .median()
            .sort_index()
            .dropna()
        )
        max_lags_t = max(min(30, len(tier_ret) // 2), 1) if len(tier_ret) >= 10 else 0
        if max_lags_t >= 1:
            plot_acf(tier_ret, lags=max_lags_t, alpha=0.05, ax=ax)
            ax.set_title(f"ACF Tier {tier_val}  (n={len(tier_ret)} days)")
        else:
            ax.text(
                0.5,
                0.5,
                f"Tier {tier_val}\nINSUFFICIENT\n(n={len(tier_ret)})",
                ha="center",
                va="center",
                transform=ax.transAxes,
                fontsize=11,
            )
    plt.suptitle("H3 — ACF(1) of log-returns per price tier")
    plt.tight_layout()
    plt.show()

=== H3: Log-Return ACF(1) by Price Tier ===

Tier 1: n=4 return observations — insufficient (need ≥5)
Tier 2: n=4 return observations — insufficient (need ≥5)
Tier 3: n=4 return observations — insufficient (need ≥5)

All tiers insufficient at n_snapshots=5.
Re-run after: approximately 2026-06-23 (≥15 daily snapshots for initial ACF).

Expected result:
  Tier 3 (>€1000): higher |ACF(1)| — speculative momentum in expensive cards
  Tier 1 (<€100):  lower |ACF(1)| — more liquid, more efficient bulk market


In [7]:
gold.close()

## 📋 Final Conclusions

```
DATA LIMITATIONS
─────────────────────────────────────────────────────────────────────────────
Snapshots:             3  (2026-06-04 to 2026-06-06)
Market log-returns:    n=2 (3 snapshots → 2 differences)
All test results:      NOT TESTABLE with current data

H1 — DISTRIBUTION STABILITY OVER TIME
─────────────────────────────────────────────────────────────────────────────
KS D:                  N/A — INSUFFICIENT DATA
  First half: 1 day, second half: 2 days (need ≥15 per half)
  With 3 snapshots, per-card "median in period" equals the single observed price —
  the two halves overlap and share almost all variance.
Stability verdict:     CANNOT ASSESS
Drift concern:         UNKNOWN — deferred

Temporal split decision (independent of H1):
  ALWAYS use temporal split — training on future data introduces target leakage
  regardless of whether the distribution is stable.
  Standard 80/20 temporal split (oldest 80% → train, newest 20% → test) is safe.

Re-run after: approximately 2026-07-04 (≥30 daily snapshots → ≥15 per half)

H2 — LOG-RETURN AUTOCORRELATION (MARKET-WIDE)
─────────────────────────────────────────────────────────────────────────────
n log-return observations:  2  (min required: 50)
max_lag (n//5 rule):        0  → no testable lags
ACF(1):                     unreliable at n=2 (all log-returns are 0.0 —
                             market median is identical across all 3 snapshots)
Ljung-Box:                  SKIPPED

Feature price_change_1d_pct:  DEFERRED — insufficient data
Feature price_change_7d_pct:  DEFERRED — insufficient data

Re-run after: approximately 2026-07-26 (≥50 daily snapshots)

H3 — ACF(1) DIFFERS BY PRICE TIER
─────────────────────────────────────────────────────────────────────────────
All tiers: 2 return observations — insufficient (need ≥5)
Tier 1 ACF(1):  N/A
Tier 2 ACF(1):  N/A
Tier 3 ACF(1):  N/A
Tier 3 > Tier 1 (speculative momentum): CANNOT ASSESS

Re-run after: approximately 2026-06-19 (≥15 daily snapshots for initial ACF)

DECISIONS FOR MODEL
─────────────────────────────────────────────────────────────────────────────
Temporal train/test split required:     YES — always (prevents leakage)
Feature price_change_1d_pct:           DEFERRED — re-evaluate 2026-07-26
Feature price_change_7d_pct:           DEFERRED — re-evaluate 2026-07-26
Separate autoregression for Tier 3:    DEFERRED — requires H3 result
Implication if H2 confirmed later:     add price_change_1d_pct and price_change_7d_pct
                                       as lag features; use rolling window = 7 days minimum

⚠ BLOCKING DEPENDENCY — 02_stationarity.ipynb (Stat 02)
─────────────────────────────────────────────────────────────────────────────
Stat 02 has a provisional, unconfirmed decision: model log-RETURNS (I(1) assumed)
vs log-LEVELS (I(0)). This decision is blocked until ≥30 snapshots (~2026-07-04).

Impact on this notebook (CDA 04) and the whole feature-engineering pipeline:
  • H1, H2, H3 test log-return properties. If Stat 02 confirms I(0) (stationary
    levels), the correct target is log-levels, not returns — and these hypotheses
    become moot for the feature-engineering phase.
  • EDA 04 ranks features by MI/correlation with log1p(EUR) LEVELS. If I(1) is
    confirmed, that ranking is NOT directly transferable to a log-return model.
    Static card attributes (edhrec_rank, rarity) explain price levels well but
    may explain short-term price changes poorly.
  • Once Stat 02 resolves: re-run EDA 04 with return-based MI before finalising
    the feature list in model_preparation/02.

RETEST SCHEDULE
─────────────────────────────────────────────────────────────────────────────
ACF/PACF visual:     2026-06-14  (≥10 snapshots)
H3 tier ACF:         2026-06-19  (≥15 snapshots)
H1 KS stability:     2026-07-04  (≥30 snapshots, ≥15 per half)
H2 Ljung-Box:        2026-07-26  (≥50 snapshots)
Stat 02 blocker:     2026-07-04  (≥30 snapshots — I(0) vs I(1) resolved)
```